# Make YOUR cloned voice (free, no account needed)

**Read these 6 sentences at a natural pace — don't rush, each takes ~12-18s.** These are the SAME 6 the notebook clones in your voice, so real and cloned clips match exactly. Record 3 English first, then 3 Hindi, one line = one clip:

1. My account number is three two seven four one nine, and I am calling to verify the mailing address currently on file, because I recently moved to a new apartment and I want to make sure nothing gets sent to the old one.
2. Could you please confirm the exact delivery time for my package tomorrow afternoon? I have a meeting until three, so I would prefer if the courier could come sometime after four, if that is possible to arrange.
3. I would like to formally report an issue with my recent bank transaction — the amount deducted does not match what I authorized, and I need this resolved before the end of the week.
4. मेरा खाता नंबर तीन दो सात चार एक नौ है, और मैं यह सुनिश्चित करना चाहता हूँ कि फ़ाइल पर मौजूद पता सही है, क्योंकि मैं हाल ही में एक नए घर में शिफ्ट हुआ हूँ।
5. क्या आप कृपया बता सकते हैं कि कल दोपहर मेरे पार्सल की डिलीवरी ठीक किस समय होगी? मुझे शाम चार बजे तक एक मीटिंग है, इसलिए उसके बाद आना ज़्यादा सही रहेगा।
6. मैं अपने हाल के बैंक लेनदेन को लेकर एक गंभीर समस्या की रिपोर्ट करना चाहता हूँ — जो राशि काटी गई है वह मेरी अनुमति से मेल नहीं खाती, और मुझे यह जल्द से जल्द ठीक करवाना है।

**Setup:** Runtime → Change runtime type → **T4 GPU** (don't skip — CPU is very slow).
Then run cells **1→2→3→4→5** in order. Cell **2** uploads your TWO reference clips (1 English + 1 Hindi); cell **3** clones each of the 6 sentences.

**Formats:** reference clips may be `.wav`, `.m4a`, `.aac`, or `.mp3` — cell **2** auto-converts them to 22050 Hz mono WAV for XTTS. Your 6 recordings are auto-converted and named `real_english_0X` / `real_hindi_0X` in cell **4**, so you never lose track of which clip is which language.


In [ ]:
# @title 1. Install XTTS-v2 dependencies
# We upgrade transformers to ensure 'is_torch_greater_or_equal' is available
!pip install -q --upgrade coqui-tts transformers

# Emergency fix: If the library still complains about MPS, we manually add the missing function
import transformers.pytorch_utils
import torch
if not hasattr(transformers.pytorch_utils, 'isin_mps_friendly'):
    transformers.pytorch_utils.isin_mps_friendly = torch.isin

# The following check attempts to import TTS. If it fails due to the previous install,
# we restart the kernel automatically.
try:
    import TTS
    print("coqui-tts OK - module `TTS` importable")
except ImportError:
    print("Updating libraries... Restarting session to apply changes. Please run this cell again after restart.")
    import os; os._exit(0)

In [ ]:
# @title 2. Set your name + upload TWO reference voices (English + Hindi)
# IMPORTANT: your real recordings MUST have said the exact sentences in cell 2 below.
# Check GPU before continuing: if the next line prints "GPU: False", first do
# Runtime -> Change runtime type -> T4 GPU, then re-run this cell.
import torch
print("GPU:", torch.cuda.is_available())
import os, pathlib, glob, shutil, re, subprocess

YOUR_NAME = "team_member_name"   # <-- CHANGE this to your name (no spaces)

os.makedirs(f"/content/clones/{YOUR_NAME}", exist_ok=True)

from google.colab import files

def upload_one(prompt):
    # Colab's popup allows multi-select; keep asking until EXACTLY one file is chosen.
    while True:
        print(prompt)
        used = files.upload()
        if len(used) == 1:
            return list(used.keys())[0]
        print(f"[ATTENTION] You selected {len(used)} file(s). The popup takes ONLY 1 - please upload exactly one file and nothing else.\n")

def to_wav(src):
    # Convert any audio (wav/m4a/aac/mp3/ogg) to 22050 Hz MONO PCM WAV - the format XTTS
    # expects for a speaker_wav reference. skips if already a clean wav (kept simple: always convert).
    dst = os.path.splitext(src)[0] + "_ref.wav"
    subprocess.run(["ffmpeg", "-y", "-i", src, "-ar", "22050", "-ac", "1",
                    "-c:a", "pcm_s16le", dst], check=True, capture_output=True)
    return dst

# English reference (one of your English clips, e.g. real_english_01.m4a)
EN_REF = upload_one("STEP 1/2. Upload ONLY 1 file: your ENGLISH reference clip (e.g. real_english_01.m4a).")
EN_REF = to_wav(EN_REF)
print("English reference set (converted to WAV):", EN_REF)

# Hindi reference (one of your Hindi clips, e.g. real_hindi_01.m4a)
HI_REF = upload_one("STEP 2/2. Upload ONLY 1 file: your HINDI reference clip (e.g. real_hindi_01.m4a).")
HI_REF = to_wav(HI_REF)
print("Hindi reference set (converted to WAV):", HI_REF)

In [ ]:
# @title 3. Generate your cloned clips (same sentences, in your voice)
# Each clone matches one sentence -> real + cloned pairs with the SAME words.
# NOTE: if you did NOT read these exact sentences in your real recording, later
# real-vs-cloned pairing will be broken. Re-record real clips to match them.
import os, glob
os.environ["COQUI_TOS_AGREED"] = "1"   # bypass first-run license prompt (would hang in Colab)
import torch
from TTS.api import TTS

# Clear any previous clones (old naming or stale reruns) so the folder never mixes old+new.
for f in glob.glob(f"/content/clones/{YOUR_NAME}/*.wav"):
    os.remove(f)

tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to("cuda" if torch.cuda.is_available() else "cpu")

sentences = [
    # English (3)
    "My account number is three two seven four one nine, and I am calling to verify the mailing address currently on file, because I recently moved to a new apartment and I want to make sure nothing gets sent to the old one.",
    "Could you please confirm the exact delivery time for my package tomorrow afternoon? I have a meeting until three, so I would prefer if the courier could come sometime after four, if that is possible to arrange.",
    "I would like to formally report an issue with my recent bank transaction — the amount deducted does not match what I authorized, and I need this resolved before the end of the week.",
    # Hindi (3)
    "मेरा खाता नंबर तीन दो सात चार एक नौ है, और मैं यह सुनिश्चित करना चाहता हूँ कि फ़ाइल पर मौजूद पता सही है, क्योंकि मैं हाल ही में एक नए घर में शिफ्ट हुआ हूँ।",
    "क्या आप कृपया बता सकते हैं कि कल दोपहर मेरे पार्सल की डिलीवरी ठीक किस समय होगी? मुझे शाम चार बजे तक एक मीटिंग है, इसलिए उसके बाद आना ज़्यादा सही रहेगा।",
    "मैं अपने हाल के बैंक लेनदेन को लेकर एक गंभीर समस्या की रिपोर्ट करना चाहता हूँ — जो राशि काटी गई है वह मेरी अनुमति से मेल नहीं खाती, और मुझे यह जल्द से जल्द ठीक करवाना है।",
]

# Clone filenames carry the language so real<->clone pairing is never ambiguous:
#   clone_english_01..03.wav  (sentences 1-3)  and clone_hindi_01..03.wav (sentences 4-6)
counts = {'english': 0, 'hindi': 0}
for idx, text in enumerate(sentences, start=1):
    lang = 'english' if idx <= 3 else 'hindi'
    counts[lang] += 1
    out = f"/content/clones/{YOUR_NAME}/clone_{lang}_{counts[lang]:02d}.wav"
    # Use the matching-language reference: English clip for EN clones, Hindi clip for HI.
    # A voice sounds a bit different per language, so an English-only ref makes the Hindi
    # clones sound 'off'. Matching refs -> better-quality Hindi clones (our differentiator).
    ref = EN_REF if idx <= 3 else HI_REF
    tts.tts_to_file(
        text=text,
        speaker_wav=ref,
        language='en' if lang == 'english' else 'hi',
        file_path=out,
    )
    print("wrote", out)

print("\nDone. Files:")
for f in sorted(glob.glob(f"/content/clones/{YOUR_NAME}/*.wav")):
    print("  ", f)

In [ ]:
# @title 4. Upload + convert your 6 REAL recordings -> WAV, auto-named by language
# Upload in TWO stages: first your 3 ENGLISH clips, then your 3 HINDI clips (multi-select
# each time). Language is taken from which stage you upload in, so it is always correct.
# Browser duplicate suffixes like " (1)" are only cosmetic and ignored.
# Output: /content/real_recs/<YOUR_NAME>/english/real_english_0X.wav + hindi/real_hindi_0X.wav
import os, subprocess
from google.colab import files

out_root = f"/content/real_recs/{YOUR_NAME}"

def upload_stage(prompt):
    while True:
        print(prompt)
        used = files.upload()
        if len(used) == 3:
            return list(used.keys())
        print(f"[ATTENTION] You selected {len(used)} file(s) - this stage needs exactly 3. Please re-select.\n")

print("We'll take your 6 recordings in two groups of 3.")
en_names = upload_stage("STEP 1/2. Upload your 3 ENGLISH clips (multi-select all 3).")
hi_names = upload_stage("STEP 2/2. Upload your 3 HINDI clips (multi-select all 3).")

# each entry: (original_src, lang, sentence_index 1-6)
tagged = [(n, 'english', i + 1) for i, n in enumerate(en_names)]
tagged += [(n, 'hindi', i + 4) for i, n in enumerate(hi_names)]

# Build output names by sorting within each language by sentence index
plan = []
for lang in ('english', 'hindi'):
    for rank, (src, ln, idx) in enumerate(sorted([t for t in tagged if t[1] == lang], key=lambda t: t[2]), start=1):
        plan.append((src, ln, f"real_{lang}_{rank:02d}.wav"))

print("\nConversion plan:")
for src, lang, out_n in plan:
    print(f"  {src:<30} -> {lang}/{out_n}")

ok = input("\nCorrect? Type y to convert, or no to abort and try again. [y/N]: ").strip().lower()
if ok != 'y':
    print("Aborted - nothing was written. Re-run this cell.")
else:
    for src, lang, out_n in plan:
        d = os.path.join(out_root, lang)
        os.makedirs(d, exist_ok=True)
        dst = os.path.join(d, out_n)
        subprocess.run(["ffmpeg", "-y", "-i", src, "-ar", "22050", "-ac", "1",
                        "-c:a", "pcm_s16le", dst], check=True, capture_output=True)
        print("wrote", dst)
    print("\nDone. Copy this folder locally then push to Drive:")
    print(f"  {out_root}")

In [ ]:
# @title 5. Download your cloned + real clips (deliver to the dataset)
from google.colab import files
import glob, os
print("Clones (named clone_english_0X / clone_hindi_0X):")
for f in sorted(glob.glob(f"/content/clones/{YOUR_NAME}/*.wav")):
    files.download(f)

real_dirs = glob.glob(f"/content/real_recs/{YOUR_NAME}/*/*.wav")
if real_dirs:
    print("\nReal clips (auto-named by language):")
    for f in sorted(real_dirs):
        files.download(f)
else:
    print("\nNo real clips found yet - run cell 4 (convert real recordings) first, then re-run this cell.")

print("\nDeliverable folder to copy locally then push to Drive:")
print(f"  /content/real_recs/{YOUR_NAME}/   (real clips)")
print(f"  /content/clones/{YOUR_NAME}/      (clone clips)")